# MNIST Digit Classification — MLP vs CNN

This notebook is a portfolio refactor of recovered university coursework. The original project compared dense MLPs and CNNs, then compared Adam with SGD learning-rate settings. The portfolio version uses a validation split drawn only from the training data so the test set remains untouched until final evaluation.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 1. Load MNIST and create train/validation/test sets

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train_full = x_train_full.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full,
    test_size=0.10,
    random_state=SEED,
    stratify=y_train_full,
)

print('Training:', x_train.shape)
print('Validation:', x_val.shape)
print('Test:', x_test.shape)

The original coursework used the MNIST test set as `validation_data` during training. For a cleaner portfolio workflow, this version creates validation data only from the original training set.

In [ ]:
x_train_dense = x_train.reshape((-1, 28 * 28))
x_val_dense = x_val.reshape((-1, 28 * 28))
x_test_dense = x_test.reshape((-1, 28 * 28))

x_train_cnn = x_train[..., np.newaxis]
x_val_cnn = x_val[..., np.newaxis]
x_test_cnn = x_test[..., np.newaxis]

y_train_cat = keras.utils.to_categorical(y_train, 10)
y_val_cat = keras.utils.to_categorical(y_val, 10)
y_test_cat = keras.utils.to_categorical(y_test, 10)

## 2. Model definitions

In [ ]:
def mlp_baseline():
    return keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax'),
    ])

def mlp_improved():
    return keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(256, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax'),
    ])

def cnn_baseline():
    return keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax'),
    ])

def cnn_improved():
    return keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3,3), activation='relu'),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(10, activation='softmax'),
    ])

## 3. Recovered assessment results

These are the recorded outputs from the recovered final notebook. They are included for historical transparency; rerun the refactored workflow before treating them as current portfolio-run metrics.

In [ ]:
recovered = pd.DataFrame({
    'Model': [
        'MLP Baseline', 'MLP Improved', 'CNN Baseline', 'CNN Improved',
        'CNN + SGD (0.01)', 'CNN + SGD (0.005)', 'CNN + Adam'
    ],
    'Test accuracy': [0.9764, 0.9776, 0.9849, 0.9910, 0.9888, 0.9873, 0.9918]
})
recovered.sort_values('Test accuracy', ascending=False)

## 4. Run the leakage-safe portfolio version

For the complete experiment, use `src/train_models.py`. It trains all seven model configurations and saves a fresh local results table, confusion matrix and classification report.